# Tavily Web Search
- LLM에게 웹검색을 쥐어주자

In [1]:
%pip install -q langchain-tavily


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
from pprint import pprint

load_dotenv()

True

In [3]:
from langchain_tavily import TavilySearch   

search_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [4]:
pprint(search_tool.invoke({'query': '강남 신세계에서 밥먹을만한 곳 알려줘'}))

{'answer': None,
 'follow_up_questions': None,
 'images': [],
 'query': '강남 신세계에서 밥먹을만한 곳 알려줘',
 'request_id': '27d866d6-5c27-4ffa-9dec-886d87e79a54',
 'response_time': 0.73,
 'results': [{'content': '강남신세계 혼밥맛집 (106곳) · 1. 소녀방앗간 서울고속터미널점 · 2. 타코벨 '
                         '서울고속버스터미널점 · 3. 소이연남 파미에스테이션점 · 4. 갓덴스시 센트럴시티점 · 5. '
                         '101번지',
              'raw_content': None,
              'score': 0.72231215,
              'title': "'강남신세계 혼밥' 맛집 빅데이터 추천순위 Top100 - 다이닝코드",
              'url': 'https://www.diningcode.com/list.dc?query=%EA%B0%95%EB%82%A8%EC%8B%A0%EC%84%B8%EA%B3%84%20%ED%98%BC%EB%B0%A5'},
             {'content': 'Missing: 밥 알려줘',
              'raw_content': None,
              'score': 0.71948266,
              'title': '강남 신세계 푸드코트 하우스오브 신세계,스위트파크(Eng CC)/ Seoul ...',
              'url': 'https://www.youtube.com/watch?v=wRvUnu8OwWI'},
             {'content': 'Missing: 밥 알려줘',
              'raw_content': None,
              'score': 0.694241,
  

In [5]:
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

llm = ChatOpenAI(model='gpt-4.1-nano')

prompt = PromptTemplate(
    input_variables=['query', 'search_results'],
    template="""넌 웹 검색결과를 요약해서 사용자의 질문에 맞게 좋은 답을 해주는 챗봇이야.

    질문: {query}
    검색결과: {search_results}
    질문에 맞는 답을 검색결과 바탕으로 간단하게 알려줘
    """
)

chain = (
    {
        'query': RunnablePassthrough(),  # 사용자 입력 그대로 이 자리에 들어옴
        'search_results': search_tool | RunnableLambda(
            lambda x: '\n'.join(
                [f'-{r['title']} ({r['content']})' for r in x['results']]
            )
        )
    }
    | prompt
    | llm
    | StrOutputParser()
)

chain.invoke('강남 신세계에서 점심 먹기 좋은곳')

"강남 신세계백화점 내에서 점심을 즐기기 좋은 곳으로는 일본 라멘 전문점인 '타치바나'가 추천됩니다. 평일 저녁에는 웨이팅이 있으나, 점심시간에는 비교적 이용이 수월할 수 있습니다. 또한, 하우스오브신세계라는 푸드코트에서도 다양한 맛집을 즐기실 수 있습니다."

In [6]:
dumb_chain = prompt | llm | StrOutputParser()

dumb_chain.invoke({'query': '강남 신세계에서 점심먹기 좋은곳', 'search_results': ''})

'강남 신세계에서 점심 맛집으로는 다양한 레스토랑이 있는데, 대표적으로는 한식, 일식, 중식 등 선택의 폭이 넓어요. 특히 신세계백화점 내부에 위치한 프리미엄 레스토랑이나 라운지에서 깔끔한 점심을 즐기기 좋습니다. 구체적인 추천으로는 한식당이나 현대적인 일식집, 또는 퓨전 레스토랑도 고려해보세요.'

In [7]:
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from datetime import datetime

today = datetime.today().strftime('%D')

search_tool = TavilySearch(
    max_results=5,
    topic='general'
)

prompt = ChatPromptTemplate.from_messages([
    ('system', f'너는 훌륭한 어시스턴트야. 반드시 웹 검색도구를 사용해서 정보를 얻어. 최대한 답을 잘 해보자. 오늘은 {today}야.'),
    MessagesPlaceholder(variable_name='messages'),
    MessagesPlaceholder(variable_name='agent_scratchpad')  # 도구(검색) 호출때 필요함
])

agent = create_openai_tools_agent(
    llm=llm,
    tools=[search_tool],
    prompt=prompt
)

agent_executor = AgentExecutor(agent=agent, tools=[search_tool], verbose=True)

agent_executor.invoke({'messages': [{'role': 'human', 'content': '오늘 날씨에 맞는 음악 추천해줘'}]})



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': '오늘 날씨에 맞는 음악 추천'}`


{'query': '오늘 날씨에 맞는 음악 추천', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://brunch.co.kr/@classictime/8', 'title': 'Siri야, 오늘 날씨에 어울리는 클래식 음악 추천해줘 - 브런치', 'content': '몇 개의 작품을 직접 골라 계절별로 날씨와 어울리는 클래식 음악을 추천하고자 하는데, 오늘은 그 중, 곧 다가올 따뜻한 봄을 맞이하며 들을 수 있는 봄을 주제로 한 곡을', 'score': 0.7380092, 'raw_content': None}, {'url': 'https://www.joongang.co.kr/article/25356747', 'title': '“오늘 날씨에 맞는 노래 틀어줘”…이러면 알아서 해주는 SKT AI', 'content': "예를 들어 사용자가 '오늘 날씨에 어울리는 노래 추천해주고 모두 틀어줘'라고 요청하면 에이닷이 날씨 정보를 확인하고 음악 추천을 수행한 후 사용자", 'score': 0.6897452, 'raw_content': None}, {'url': 'https://www.koreadaily.com/article/20250804085158028', 'title': '“오늘 날씨에 맞는 노래 틀어줘”…이러면 알아서 해주는 SKT AI', 'content': "예를 들어 사용자가 '오늘 날씨에 어울리는 노래 추천해주고 모두 틀어줘'라고 요청하면 에이닷이 날씨 정보를 확인하고 음악 추천을 수행한 후 사용자", 'score': 0.6897452, 'raw_content': None}, {'url': 'https://www.reddit.com/r/musicsuggestions/co

{'messages': [{'role': 'human', 'content': '오늘 날씨에 맞는 음악 추천해줘'}],
 'output': '오늘 날씨에 맞는 음악 추천을 위해 검색한 결과, 따뜻하고 맑은 날씨에 어울리는 음악들이 추천되고 있는데요. 예를 들어, 클래식 음악이나 부드러운 기타 라인의 곡들이 적합하다고 합니다. 특히, 따뜻한 봄을 맞이하며 듣기 좋은 곡이나, 맑은 날씨에 어울리는 분위기의 음악들이 추천되고 있습니다. \n\n혹시 특정 스타일이나 기분을 원하신다면 더 구체적으로 알려주시면 맞춤 추천도 도와드릴 수 있어요!'}